# [KAMP] 주제 ④ X-ray 완제품 이물질 탐지 및 AI 미탐지 조건 분석
## 엔드투엔드 파이프라인 마스터 튜토리얼 (YOLOv8 기반)

---
### 📌 과제 개요 및 심사 배점표 (100점 만점)
1. **데이터 진단 (15점)**: 데이터 무결성 검증, 파일 해시 확인, 그룹 단위 누수 방지 분할
2. **모델 개발 (40점)**: YOLOv8 기반 고정밀 이물질 탐지 모델 구축 및 베이스라인 비교
3. **영향요인·오류분석 (15점)**: 크기/위치/대비 조건별 미탐(FN) 원인 분석 및 실패 사례 시각화
4. **현장 활용성 (10점)**: 안전을 고려한 2단계 판정 임계값 및 3-Tier 재검사 SOP 수립
5. **창의성 (10점)**: 지름길 학습(Shortcut Learning) 방지를 위한 순수 색채 인페인팅
6. **재현성 (10점)**: 원클릭 재현 가능한 모듈형 코드 및 완전한 파이프라인
---


### 1. 환경 설정 및 전체 프로젝트 자동 연동
KAMP Note 환경에서 프로젝트를 자동 감지(또는 단독 업로드 시 GitHub에서 원클릭 동기화)하고, Tesla V100 GPU 호환 패키지를 일괄 설치합니다.


In [ ]:
import os, sys

# 1) Google Colab 환경 감지 및 구글 드라이브 마운트
if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print('[*] Google Colab 감지: 구글 드라이브를 마운트합니다...')
        drive.mount('/content/drive')

# 2) 프로젝트 루트 디렉토리 자동 감지 및 작업 경로(Working Directory) 이동
candidates = [
    os.getcwd(),
    '/content/drive/MyDrive/kamp_xray_pipeline',
    '/content/drive/MyDrive/kamp-project',
    '/content/drive/MyDrive/kamp',
    '/content/kamp-project',
    '/content/kamp_xray_pipeline',
    os.path.abspath(os.path.join(os.getcwd(), 'kamp-project')),
    os.path.abspath(os.path.join(os.getcwd(), 'kamp_xray_pipeline')),
    os.path.abspath(os.path.join(os.getcwd(), '..')),
    os.path.expanduser('~/kamp-project'),
    os.path.expanduser('~/kamp_xray_pipeline'),
]

root_found = False
for cand in candidates:
    if os.path.exists(os.path.join(cand, 'scripts', '00_audit_data.py')):
        %cd $cand
        if cand not in sys.path:
            sys.path.insert(0, cand)
        print(f'[*] 프로젝트 루트 경로 설정 완료: {os.getcwd()}')
        root_found = True
        break

# 만약 드라이브나 로컬에 폴더가 없으면 GitHub에서 전체 프로젝트 자동 클론
if not root_found:
    print('[*] 프로젝트 폴더가 감지되지 않아 GitHub에서 최신 파이프라인 전체를 다운로드합니다...')
    !git clone https://github.com/opiniom/kamp-project.git
    if os.path.exists('kamp-project'):
        %cd kamp-project
        if os.getcwd() not in sys.path:
            sys.path.insert(0, os.getcwd())
        print(f'[*] GitHub 클론 후 루트 이동 완료: {os.getcwd()}')
        root_found = True

# 3) 필수 라이브러리 일괄 설치 (Colab 및 KAMP Note 완벽 호환)
!pip install -q opencv-python-headless pyyaml pandas matplotlib ultralytics


In [ ]:
import os, sys, glob, json, time, shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch
from ultralytics import YOLO

print(f"[*] Python 버전: {sys.version.split()[0]}")
print(f"[*] PyTorch 버전: {torch.__version__} (CUDA 사용가능: {torch.cuda.is_available()})")
if torch.cuda.is_available():
    print(f"[*] GPU 장치명: {torch.cuda.get_device_name(0)}")


### 2. [단계 A] 데이터 진단 및 전수 감사 (데이터 진단 15점)
- 원본 이미지와 500개 공식 TXT 라벨의 1:1 매칭 및 무결성 검증
- 실제 파일 포맷(매직 바이트), 해상도, SHA-256 및 픽셀 해시 산출
- `scripts/00_audit_data.py` 실행


In [ ]:
# 데이터 감사 스크립트 실행
!python scripts/00_audit_data.py

# 생성된 manifest.csv 확인
df_manifest = pd.read_csv('manifests/manifest.csv')
print(f"총 등록된 이미지 수: {len(df_manifest)}장")
df_manifest[['base_name', 'machine', 'date', 'width', 'height', 'num_boxes']].head()


### 3. [단계 B] 그룹 누수 방지 데이터 분할 (Group-aware Split)
- 동일 제품 또는 연속 촬영본이 Train/Val/Test에 섞이는 Data Leakage 원천 차단
- 호기 및 촬영일자(`group_id`) 단위 60/20/20 분할
- `scripts/01_make_splits.py` 실행


In [ ]:
# 그룹 분할 스크립트 실행
!python scripts/01_make_splits.py

import json
with open('manifests/split_audit.json', 'r', encoding='utf-8') as f:
    split_info = json.load(f)

print(f"[분할 현황] Train: {split_info['train_count']}장, Val: {split_info['val_count']}장, Test: {split_info['test_count']}장")
print(f"[누수 검증] 그룹 교집합: {split_info['group_leakage_count']}건, 픽셀 중복: {split_info['pixel_leakage_count']}건")


### 4. [단계 C] 색상 표시 인페인팅 전처리 (창의성 10점)
- 원본 픽셀에 박혀있는 인위적 색상 박스 노이즈(지름길 학습 유발)를 순수 채도(Saturation) 마스킹으로 감지
- OpenCV Telea Inpainting으로 테두리 선만 주변 흑백 질감으로 복원 (정답 TXT 미참조로 정답 누수 차단)
- `scripts/02_preprocess.py` 실행


In [ ]:
# 인페인팅 전처리 및 YOLOv8 데이터셋 구축
!python scripts/02_preprocess.py

# 전처리 감사 리포트 확인
df_prep = pd.read_csv('manifests/preprocessing_audit.csv')
print(f"전처리 적용률: {df_prep['inpaint_applied'].mean()*100:.1f}%")
print(f"평균 색상 마스크 면적 비율: {df_prep[df_prep['inpaint_applied']]['mask_ratio_pct'].mean():.3f}%")


### 5. [단계 D] YOLOv8 모델 파인튜닝 학습 (모델 개발 40점, 조건 1 충족)
- 최신 Anchor-free 아키텍처 YOLOv8s 모델을 인페인팅 전처리 데이터셋에 파인튜닝
- `scripts/03_train.py` 실행 (Tesla V100 GPU 환경에서 30 에폭 기준 약 1~2분 소요)


In [ ]:
# 모델 학습 실행 (Tesla V100 고성능 GPU 기준 30 에폭 1~2분 완료)
!python scripts/03_train.py --epochs 30 --batch 16 --name kamp_yolov8_final


### 6. [단계 E] 독립 Test 세트 평가 및 기존 베이스라인 비교
- 고정 IoU 0.5 평가 프로토콜 하에서 Precision, Recall, F1-Score, mAP, FPS 산출
- FROC (Free-response ROC: x=FP/Image, y=Recall) 곡선 생성
- `scripts/04_predict_and_evaluate.py` 실행


In [ ]:
# 평가 스크립트 실행
!python scripts/04_predict_and_evaluate.py

import json
with open('reports/evaluation_metrics.json', 'r', encoding='utf-8') as f:
    eval_res = json.load(f)

print("[독립 Test 세트 평가 성과]")
print(f"- Precision : {eval_res['precision']*100:.2f}%")
print(f"- Recall    : {eval_res['recall']*100:.2f}%")
print(f"- F1-Score  : {eval_res['f1_score']:.4f}")
print(f"- 추론 속도 : {eval_res['mean_latency_ms']} ms ({eval_res['fps']} FPS)")


### 7. [단계 F] 조건별 미탐 성능 분석 (영향요인·오류분석 15점, 조건 2 충족)
- 이물질 크기(초소형/중형/대형), 위치(제품 가장자리 vs 중심부), 명암 대비별 검출률(Recall) 정량 분석
- `scripts/05_condition_analysis.py` 실행


In [ ]:
# 조건별 미탐 분석 실행
!python scripts/05_condition_analysis.py

import json
with open('reports/condition_analysis_report.json', 'r', encoding='utf-8') as f:
    cond_res = json.load(f)

print("[조건별 검출률 분석 요약]")
print("1. 크기별 분석:", cond_res['size_analysis'])
print("2. 위치별 분석:", cond_res['location_analysis'])
print("3. 대비별 분석:", cond_res['contrast_analysis'])


### 8. [단계 G] 안전 임계값 도출 및 3-Tier 재검사 SOP (현장 활용 10점, 조건 3, 4 충족)
- 미탐지 위험 비용을 반영한 Cost Matrix 최적화 기반 2단계 안전 임계값(tau_low, tau_high) 도출
- 3-Tier(불합격 / 90도 회전 재촬영 재검사 / AI 미경고) 표준작업절차서(SOP) 자동 수립
- `scripts/06_threshold_policy.py` 실행


In [ ]:
# 안전 임계값 및 SOP 산출 실행
!python scripts/06_threshold_policy.py

# 생성된 SOP 텍스트 확인
with open('reports/standard_operating_procedure.txt', 'r', encoding='utf-8') as f:
    print(f.read())
